In [1]:
import pandas as pd
import numpy as np
import pyreadstat
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test, multivariate_logrank_test
from lifelines.plotting import add_at_risk_counts
# ── Dossiers de sortie ────────────────────────────────────
os.makedirs("../outputs/figures", exist_ok=True)
os.makedirs("../outputs/tables",  exist_ok=True)

# ── Style graphiques ──────────────────────────────────────
plt.rcParams['figure.dpi']        = 130
plt.rcParams['font.family']       = 'DejaVu Sans'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid')
NAVY  = "#0D1B4B"
BLUE  = "#2196F3"
GREEN = "#388E3C"
CORAL = "#FF5722"
AMBER = "#F57C00"
GRAY  = "#90A4AE"

print("✓ Toutes les bibliothèques chargées !")
print(f"  lifelines disponible pour : Kaplan-Meier · Log-rank · Cox")

✓ Toutes les bibliothèques chargées !
  lifelines disponible pour : Kaplan-Meier · Log-rank · Cox


In [3]:
# ── 1.1 Chargement du fichier enfants ────────────────────
print("Chargement de donnees_survie_premiers_nes_bcg.csv...")
df_raw = pd.read_csv("../donnees_survie_premiers_nes_bcg.csv")
df_raw.columns = df_raw.columns.str.lower()

print(f"✓ Fichier chargé : {df_raw.shape[0]:,} enfants · {df_raw.shape[1]} variables")
print(f"\nAperçu des premières colonnes :")
print(list(df_raw.columns[:30]))

Chargement de donnees_survie_premiers_nes_bcg.csv...
✓ Fichier chargé : 4,792 enfants · 10 variables

Aperçu des premières colonnes :
['bidx', 'b3', 'b5', 'v008', 'h2', 'h2m', 'h2y', 'bcg_cmc', 'evenement_bcg', 'temps_survie']


In [4]:
# ── 1.1b Enrichissement avec covariables depuis CMIR71FL.SAV ─
# Stratégie : h2/survie vient du CSV existant ; covariables (v025,v106,v190,v024,v012,b4)
# viennent du SAV, liées via la clé (bidx, b3, v008).
import re
print("Extraction des covariables depuis CMIR71FL.SAV...")

# 1. Lire les noms de colonnes ORIGINAUX
_, meta_ir = pyreadstat.read_sav("../data/CMIR71FL.SAV", row_limit=1)
cols_orig  = list(meta_ir.column_names)
cols_lower = [str(c).lower() for c in cols_orig]
low2orig   = {lo: orig for lo, orig in zip(cols_lower, cols_orig)}

# 2. Détecter le format des colonnes de naissance (ex : b3$01)
b3_tous = sorted([c for c in cols_lower if c.startswith('b3') and len(c) > 2])
if not b3_tous:
    print("⚠ Aucune colonne b3... dans le SAV — vérifier le fichier")
    raise SystemExit("Fichier inattendu")

ex  = b3_tous[0]
m   = re.match(r'b3([^0-9]*)(\d+)$', ex)
sep = m.group(1) if m else '_'
pad = len(m.group(2)) if m else 2
def make_sfx(i): return f'{sep}{str(i).zfill(pad)}'
print(f"  Convention : '{ex}'  →  sep='{sep}', pad={pad}  ⇒  b3{make_sfx(1)}, bord{make_sfx(1)}, ...")

# 3. Colonnes à charger : femme + identification naissance (SANS h2 — absent du SAV)
cols_mere_lo = ['v001','v002','v003','v008','v012','v024','v025','v106','v190']
cols_enf_lo  = []
for i in range(1, 21):
    s = make_sfx(i)
    for var in ['b3','b4','bord','b5']:
        c = f'{var}{s}'
        if c in cols_lower:
            cols_enf_lo.append(c)

cols_load_lo   = [c for c in cols_mere_lo + cols_enf_lo if c in cols_lower]
cols_load_orig = [low2orig[c] for c in cols_load_lo]
n_m = len([c for c in cols_mere_lo if c in cols_lower])
print(f"  À charger : {len(cols_load_lo)} colonnes ({n_m} mères + {len(cols_enf_lo)} naissances)")

# 4. Charger le SAV
df_ir, _ = pyreadstat.read_sav("../data/CMIR71FL.SAV", usecols=cols_load_orig)
df_ir.columns = [str(c).lower() for c in df_ir.columns]
print(f"  ✓ {df_ir.shape[0]:,} femmes  |  {df_ir.shape[1]} colonnes")

# 5. Reshape wide → long + conserver la position (= bidx DHS : 1=naissance la + récente)
frames = []
for i in range(1, 21):
    s = make_sfx(i)
    rename = {f'b3{s}':'b3', f'b4{s}':'b4', f'bord{s}':'bord', f'b5{s}':'b5'}
    rename_ok = {k: v for k, v in rename.items() if k in df_ir.columns}
    if not rename_ok:
        continue
    cols_m = [c for c in cols_mere_lo if c in df_ir.columns]
    tmp = df_ir[cols_m + list(rename_ok.keys())].rename(columns=rename_ok).copy()
    for col in ['b3','b4','bord','b5']:
        if col not in tmp.columns: tmp[col] = np.nan
    tmp['bidx'] = i   # position dans l'historique de naissances
    frames.append(tmp)

df_long = pd.concat(frames, ignore_index=True)
df_long = df_long[df_long['b3'].notna()].copy()
print(f"  Format long : {len(df_long):,} naissances")

# 6. Conversion numérique forcée (SPSS peut renvoyer float ou string)
for col in ['bord','b3','b4','b5','v008','v012','v025','v106','v190','v024','bidx']:
    if col in df_long.columns:
        df_long[col] = pd.to_numeric(df_long[col], errors='coerce')

print(f"  bord — dtype:{df_long['bord'].dtype} | valeurs:{sorted(df_long['bord'].dropna().unique().tolist())[:8]}")

# 7. Filtrer premiers-nés (bord==1) nés ≤ 60 mois avant enquête
df_long['age_mois'] = df_long['v008'] - df_long['b3']
df_prem = df_long[
    (df_long['bord'] == 1) &
    (df_long['age_mois'].between(0, 60))
].copy()
print(f"  Premiers-nés ≤ 5 ans dans SAV : {len(df_prem):,}")

# 8. Charger le CSV existant (qui contient h2, h2m, h2y, survie)
df_csv = pd.read_csv("../donnees_survie_premiers_nes_bcg.csv")
df_csv.columns = df_csv.columns.str.lower()
for col in ['bidx','b3','v008','h2','h2m','h2y','bcg_cmc','evenement_bcg','temps_survie']:
    if col in df_csv.columns:
        df_csv[col] = pd.to_numeric(df_csv[col], errors='coerce')
print(f"  CSV chargé : {len(df_csv):,} lignes | colonnes : {list(df_csv.columns)}")

# 9. Fusion via (bidx, b3, v008) — clé quasi-unique pour chaque premier-né
# bidx = position dans historique (1=plus récent), identique dans CSV et SAV
covars_sav = [c for c in ['bidx','b3','v008','v025','v106','v190','v024','v012','b4','v001','v002','v003']
              if c in df_prem.columns]
df_sav_cov = df_prem[covars_sav].drop_duplicates(subset=['bidx','b3','v008']).copy()

# Diagnostiquer les doublons avant fusion
n_dup = df_sav_cov.duplicated(subset=['bidx','b3','v008']).sum()
print(f"\n  Clé (bidx,b3,v008) — doublons dans SAV : {n_dup}")

df_raw = df_csv.merge(
    df_sav_cov,
    on=['bidx','b3','v008'],
    how='left'
)

# Retirer les enfants h2 manquant ou ne sait pas (h2=8)
df_raw = df_raw[df_raw['h2'].notna() & (df_raw['h2'] != 8)].copy()

print(f"\n✓ df_raw enrichi : {len(df_raw):,} premiers-nés")
for var, lbl in [('v025','Résidence'),('v024','Région'),('v106','Éducation'),
                 ('v190','Richesse'),('v012','Âge mère'),('b4','Sexe enfant')]:
    if var in df_raw.columns:
        n_ok = df_raw[var].notna().sum()
        pct  = n_ok / len(df_raw) * 100 if len(df_raw) > 0 else 0
        print(f"  {var} ({lbl}) : {n_ok:,} valides ({pct:.1f}%)")
    else:
        print(f"  {var} ({lbl}) : ABSENTE")

Extraction des covariables depuis CMIR71FL.SAV...
  Convention : 'b3$01'  →  sep='$', pad=2  ⇒  b3$01, bord$01, ...
  À charger : 89 colonnes (9 mères + 80 naissances)
  ✓ 14,677 femmes  |  89 colonnes
  Format long : 33,988 naissances
  bord — dtype:float64 | valeurs:[1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
  Premiers-nés ≤ 5 ans dans SAV : 2,309
  CSV chargé : 4,792 lignes | colonnes : ['bidx', 'b3', 'b5', 'v008', 'h2', 'h2m', 'h2y', 'bcg_cmc', 'evenement_bcg', 'temps_survie']

  Clé (bidx,b3,v008) — doublons dans SAV : 0

✓ df_raw enrichi : 4,789 premiers-nés
  v025 (Résidence) : 4,669 valides (97.5%)
  v024 (Région) : 4,669 valides (97.5%)
  v106 (Éducation) : 4,669 valides (97.5%)
  v190 (Richesse) : 4,669 valides (97.5%)
  v012 (Âge mère) : 4,669 valides (97.5%)
  b4 (Sexe enfant) : 4,669 valides (97.5%)


In [5]:
# ── 1.2 Recensement des variables clés ───────────────────
VARIABLES_CIBLES = {
    # SURVIE
    'b3' : 'Date naissance enfant (CMC)',
    'b5' : 'Enfant vivant (1=oui / 0=non)',
    'b6' : 'Âge au décès (jours si <1 mois)',
    'b7' : 'Âge au décès (mois)',
    'bord': 'Rang de naissance',
    'b11' : 'Intervalle depuis naissance précédente',
    # VACCINATION
    'h0'  : 'Vaccin Polio 0 (naissance)',
    'h2'  : 'Vaccin BCG reçu',
    'h2d' : 'Jour réception BCG',
    'h2m' : 'Mois réception BCG',
    'h2y' : 'Année réception BCG',
    'h3'  : 'Vaccin DTP1/Penta1',
    # SOCIOÉCONOMIQUES
    'v008': 'Date entretien (CMC)',
    'v005': 'Poids de sondage',
    'v001': 'Numéro de cluster',
    'v012': 'Âge de la mère',
    'v025': 'Milieu de résidence (1=urbain, 2=rural)',
    'v106': "Niveau d'instruction de la mère",
    'v190': 'Quintile de richesse',
    'm15' : "Lieu d'accouchement",
    'b4'  : 'Sexe de l\'enfant (1=masculin, 2=féminin)',
    'b8'  : 'Âge actuel de l\'enfant (années)',
    'v024' : 'Région',
}

print("=" * 65)
print("RECENSEMENT DES VARIABLES CLÉS")
print("=" * 65)
print(f"{'Code DHS':<10} {'Statut':<12} {'Description'}")
print("-" * 65)

presentes, absentes = [], []
for code_var, description in VARIABLES_CIBLES.items():
    if code_var in df_raw.columns:
        n_missing = df_raw[code_var].isna().sum()
        pct_missing = n_missing / len(df_raw) * 100
        print(f"{code_var:<10} {'✓ PRÉSENTE':<12} {description[:40]} [{pct_missing:.1f}% NA]")
        presentes.append(code_var)
    else:
        print(f"{code_var:<10} {'✗ ABSENTE':<12} {description[:40]}")
        absentes.append(code_var)

print(f"\n{'='*65}")
print(f"Présentes : {len(presentes)}/{len(VARIABLES_CIBLES)} variables")
if absentes:
    print(f"Absentes  : {absentes}")
else:
    print("✓ Toutes les variables sont disponibles !")

RECENSEMENT DES VARIABLES CLÉS
Code DHS   Statut       Description
-----------------------------------------------------------------
b3         ✓ PRÉSENTE   Date naissance enfant (CMC) [0.0% NA]
b5         ✓ PRÉSENTE   Enfant vivant (1=oui / 0=non) [0.0% NA]
b6         ✗ ABSENTE    Âge au décès (jours si <1 mois)
b7         ✗ ABSENTE    Âge au décès (mois)
bord       ✗ ABSENTE    Rang de naissance
b11        ✗ ABSENTE    Intervalle depuis naissance précédente
h0         ✗ ABSENTE    Vaccin Polio 0 (naissance)
h2         ✓ PRÉSENTE   Vaccin BCG reçu [0.0% NA]
h2d        ✗ ABSENTE    Jour réception BCG
h2m        ✓ PRÉSENTE   Mois réception BCG [32.1% NA]
h2y        ✓ PRÉSENTE   Année réception BCG [32.1% NA]
h3         ✗ ABSENTE    Vaccin DTP1/Penta1
v008       ✓ PRÉSENTE   Date entretien (CMC) [0.0% NA]
v005       ✗ ABSENTE    Poids de sondage
v001       ✓ PRÉSENTE   Numéro de cluster [2.5% NA]
v012       ✓ PRÉSENTE   Âge de la mère [2.5% NA]
v025       ✓ PRÉSENTE   Milieu de résidence

In [6]:
# ── 1.3 Exploration des modalités des variables clés ─────
print("=" * 60)
print("MODALITÉS DES VARIABLES CLÉS")
print("=" * 60)

vars_explorer = {
    'b5' : 'Statut vital (event)',
    'bord': 'Rang de naissance',
    'h0'  : 'Vaccin Polio 0',
    'h2'  : 'Vaccin BCG',
    'h3'  : 'Vaccin DTP1',
    'v025': 'Milieu résidence',
    'v106': "Niveau instruction",
    'b4'  : 'Sexe enfant',
}

for var, label in vars_explorer.items():
    if var in df_raw.columns:
        counts = df_raw[var].value_counts(dropna=False).sort_index()
        print(f"\n--- {label} ({var}) ---")
        for val, cnt in counts.items():
            pct = cnt/len(df_raw)*100
            print(f"  {str(val):<8} : {cnt:>6,} ({pct:.1f}%)")

MODALITÉS DES VARIABLES CLÉS

--- Statut vital (event) (b5) ---
  1        :  4,789 (100.0%)

--- Vaccin BCG (h2) ---
  0        :    739 (15.4%)
  1        :  3,251 (67.9%)
  2        :    775 (16.2%)
  3        :     24 (0.5%)

--- Milieu résidence (v025) ---
  1.0      :  2,280 (47.6%)
  2.0      :  2,389 (49.9%)
  nan      :    120 (2.5%)

--- Niveau instruction (v106) ---
  0.0      :    685 (14.3%)
  1.0      :  1,276 (26.6%)
  2.0      :  2,147 (44.8%)
  3.0      :    561 (11.7%)
  nan      :    120 (2.5%)

--- Sexe enfant (b4) ---
  1.0      :  2,522 (52.7%)
  2.0      :  2,147 (44.8%)
  nan      :    120 (2.5%)


In [7]:
# ── 1.4 Distribution de b7 (âge au décès en mois) ───────
if 'b7' in df_raw.columns:
    b7_nonmiss = df_raw['b7'].dropna()
    deces_total = (df_raw['b5'] == 0).sum() if 'b5' in df_raw.columns else "?"
    
    print(f"Variable b7 — Âge au décès (mois) :")
    print(f"  Enfants décédés (b5=0)    : {deces_total:,}")
    print(f"  b7 renseigné (non NA)     : {len(b7_nonmiss):,}")
    print(f"  b7 = 0 (décès néonatal)   : {(b7_nonmiss==0).sum():,}")
    print(f"  b7 ≤ 1 mois               : {(b7_nonmiss<=1).sum():,}")
    print(f"  b7 ≤ 12 mois (infantile)  : {(b7_nonmiss<=12).sum():,}")
    print(f"  b7 ≤ 60 mois (< 5 ans)    : {(b7_nonmiss<=60).sum():,}")
    print(f"  b7 max                    : {b7_nonmiss.max():.0f} mois")

In [8]:
# ── 2.1 Application des critères étape par étape ─────────
print("=" * 60)
print("FLOWCHART D'INCLUSION / EXCLUSION")
print("=" * 60)

N0 = len(df_raw)
print(f"\nN0 — Tous les enfants dans CKIR71FL.SAV : {N0:,}")

# ── Critère 1 : Premiers-nés uniquement ──────────────────
if 'bord' in df_raw.columns:
    df1 = df_raw[df_raw['bord'] == 1].copy()
    excl_1 = N0 - len(df1)
    print(f"\n[-] Exclus (rang ≥ 2)              : {excl_1:,}")
    print(f"[→] N1 — Premiers-nés (bord=1)     : {len(df1):,}")
else:
    # Alternative si bord absent : utiliser bidx (birth index)
    df1 = df_raw[df_raw.get('bidx', df_raw.get('b0', pd.Series([1]*len(df_raw)))) == 1].copy()
    print(f"[→] N1 — Premiers-nés              : {len(df1):,}")

# ── Critère 2 : Nés dans les 5 ans ───────────────────────
if 'v008' in df1.columns and 'b3' in df1.columns:
    df1['age_mois_enquete'] = df1['v008'] - df1['b3']
    df2 = df1[df1['age_mois_enquete'].between(0, 60)].copy()
    excl_2 = len(df1) - len(df2)
    print(f"\n[-] Exclus (nés hors fenêtre 5 ans): {excl_2:,}")
    print(f"[→] N2 — Nés dans les 5 ans        : {len(df2):,}")
else:
    df2 = df1.copy()
    print(f"[→] N2 — (variable CMC non dispo)  : {len(df2):,}")

# ── Critère 3 : b5 non manquant ──────────────────────────
if 'b5' in df2.columns:
    df3 = df2[df2['b5'].notna()].copy()
    excl_3 = len(df2) - len(df3)
    print(f"\n[-] Exclus (b5 manquant)           : {excl_3:,}")
    print(f"[→] N3 — Statut vital connu        : {len(df3):,}")
else:
    df3 = df2.copy()
    print("[!] Variable b5 non trouvée — vérifier le fichier")

# ── Critère 4 : h2 non manquant ──────────────────────────
if 'h2' in df3.columns:
    df4 = df3[df3['h2'].notna()].copy()
    excl_4 = len(df3) - len(df4)
    print(f"\n[-] Exclus (h2/BCG manquant)       : {excl_4:,}")
    print(f"[→] N4 — Statut BCG connu          : {len(df4):,}")
else:
    df4 = df3.copy()
    print("[!] Variable h2 (BCG) non trouvée")

# ── Critère 5 : Cohérence temps ──────────────────────────
if 'b5' in df4.columns and 'b7' in df4.columns:
    # Exclure : décédé (b5=0) mais âge au décès inconnu (b7 NA)
    mask_deces_sans_age = (df4['b5'] == 0) & (df4['b7'].isna())
    excl_5 = mask_deces_sans_age.sum()
    df5 = df4[~mask_deces_sans_age].copy()
    print(f"\n[-] Exclus (décédés sans âge b7)   : {excl_5:,}")
    print(f"[→] N5 — ÉCHANTILLON FINAL         : {len(df5):,}")
else:
    df5 = df4.copy()

print(f"\n{'='*60}")
print(f"RÉSUMÉ FLOWCHART")
print(f"  Effectif initial    : {N0:,}")
print(f"  Effectif final      : {len(df5):,}")
print(f"  Total exclus        : {N0 - len(df5):,} ({(N0-len(df5))/N0*100:.1f}%)")

# Sauvegarder l'échantillon filtré
df = df5.copy()
print(f"\n✓ DataFrame final 'df' créé : {df.shape[0]:,} premiers-nés")

FLOWCHART D'INCLUSION / EXCLUSION

N0 — Tous les enfants dans CKIR71FL.SAV : 4,789
[→] N1 — Premiers-nés              : 4,789

[-] Exclus (nés hors fenêtre 5 ans): 0
[→] N2 — Nés dans les 5 ans        : 4,789

[-] Exclus (b5 manquant)           : 0
[→] N3 — Statut vital connu        : 4,789

[-] Exclus (h2/BCG manquant)       : 0
[→] N4 — Statut BCG connu          : 4,789

RÉSUMÉ FLOWCHART
  Effectif initial    : 4,789
  Effectif final      : 4,789
  Total exclus        : 0 (0.0%)

✓ DataFrame final 'df' créé : 4,789 premiers-nés


In [9]:
# ── 3.1 Variable événement (vaccination BCG) ─────────────
# event = 1 si l'enfant a reçu le BCG, 0 si censuré (non vacciné à l'enquête)
df['event'] = df['evenement_bcg'].astype(int)

print(f"Variable event (vaccination BCG) :")
print(f"  event = 1 (vacciné BCG)  : {df['event'].sum():,} ({df['event'].mean()*100:.2f}%)")
print(f"  event = 0 (censuré)      : {(df['event']==0).sum():,} ({(df['event']==0).mean()*100:.2f}%)")

Variable event (vaccination BCG) :
  event = 1 (vacciné BCG)  : 3,251 (67.88%)
  event = 0 (censuré)      : 1,538 (32.12%)


In [10]:
# ── 3.2 Variable temps T ─────────────────────────────────
# Délai en mois depuis la naissance jusqu'à la vaccination BCG (ou censure)
# La variable temps_survie est déjà calculée dans le fichier CSV
df['duree'] = df['temps_survie'].clip(lower=0.5)  # minimum 0,5 mois pour éviter les 0

print(f"Variable duree (mois jusqu'au BCG ou censure) :")
print(f"  Minimum : {df['duree'].min():.1f} mois")
print(f"  Médiane : {df['duree'].median():.1f} mois")
print(f"  Moyenne : {df['duree'].mean():.1f} mois")
print(f"  Maximum : {df['duree'].max():.1f} mois")

Variable duree (mois jusqu'au BCG ou censure) :
  Minimum : 0.5 mois
  Médiane : 1.0 mois
  Moyenne : 6.2 mois
  Maximum : 36.0 mois


In [11]:
# ── 3.4 Variable BCG (exposition principale) ──────────────
# h2 = 0 : non vacciné
# h2 = 1 : vacciné (carte)
# h2 = 2 : vacciné (déclaration mère)
# h2 = 8 : ne sait pas → traité comme NA

df['bcg'] = df['h2'].map({
    0: 0,   # Non vacciné
    1: 1,   # Vacciné (carte)
    2: 1,   # Vacciné (déclaration)
    8: np.nan  # Ne sait pas → exclu
})

df['bcg_label'] = df['bcg'].map({0: 'Non vacciné BCG', 1: 'Vacciné BCG'})

print("Variable BCG (exposition principale) :")
print(f"  Vacciné BCG (bcg=1)      : {(df['bcg']==1).sum():,} ({(df['bcg']==1).mean()*100:.1f}%)")
print(f"  Non vacciné BCG (bcg=0)  : {(df['bcg']==0).sum():,} ({(df['bcg']==0).mean()*100:.1f}%)")
print(f"  Ne sait pas (NA)         : {df['bcg'].isna().sum():,}")

# Exclure les "ne sait pas" pour l'analyse principale
df_analyse = df[df['bcg'].notna()].copy()
print(f"\n✓ Échantillon d'analyse final (BCG connu) : {len(df_analyse):,} premiers-nés")

Variable BCG (exposition principale) :
  Vacciné BCG (bcg=1)      : 4,026 (84.1%)
  Non vacciné BCG (bcg=0)  : 739 (15.4%)
  Ne sait pas (NA)         : 24

✓ Échantillon d'analyse final (BCG connu) : 4,765 premiers-nés


In [12]:
# ── 3.5 Covariables de l'analyse ─────────────────────────
# Labels pour les graphiques
if 'v025' in df_analyse.columns:
    df_analyse['residence'] = df_analyse['v025'].map({1:'Urbain', 2:'Rural'})

if 'v106' in df_analyse.columns:
    df_analyse['instruction'] = df_analyse['v106'].map({
        0:'Aucun', 1:'Primaire', 2:'Secondaire', 3:'Supérieur'
    })

if 'v190' in df_analyse.columns:
    df_analyse['quintile'] = df_analyse['v190'].map({
        1:'Très pauvre', 2:'Pauvre', 3:'Moyen', 4:'Riche', 5:'Très riche'
    })

if 'b4' in df_analyse.columns:
    df_analyse['sexe'] = df_analyse['b4'].map({1:'Masculin', 2:'Féminin'})

if 'v012' in df_analyse.columns:
    df_analyse['age_mere_cat'] = pd.cut(
        df_analyse['v012'],
        bins=[14, 19, 24, 29, 34, 49],
        labels=['15-19', '20-24', '25-29', '30-34', '35-49']
    )

print("Covariables créées :")
for var in ['residence','instruction','quintile','sexe','age_mere_cat']:
    if var in df_analyse.columns:
        n_valid = df_analyse[var].notna().sum()
        print(f"  {var:<20} : {n_valid:,} valides")

print(f"\n✓ Toutes les variables de survie sont prêtes !")
print(f"  N final pour l'analyse : {len(df_analyse):,} premiers-nés")

Covariables créées :
  residence            : 4,645 valides
  instruction          : 4,645 valides
  quintile             : 4,645 valides
  sexe                 : 4,645 valides
  age_mere_cat         : 4,645 valides

✓ Toutes les variables de survie sont prêtes !
  N final pour l'analyse : 4,765 premiers-nés


In [13]:
# ── 4.1 Tableau descriptif général ──────────────────────
N = len(df_analyse)
print("=" * 60)
print(f"TABLEAU 1 — DESCRIPTION DE L'ÉCHANTILLON FINAL (N={N:,})")
print("=" * 60)

# Statut vaccinal BCG (event = evenement_bcg)
n_vaccines     = df_analyse['event'].sum()
n_non_vaccines = (df_analyse['event'] == 0).sum()
print(f"\n1. STATUT VACCINAL BCG (événement de survie)")
print(f"   Vacciné BCG — date connue (event=1) : {n_vaccines:,} ({n_vaccines/N*100:.1f}%)")
print(f"   Censuré — date inconnue   (event=0) : {n_non_vaccines:,} ({n_non_vaccines/N*100:.1f}%)")

# Statut BCG selon h2
print(f"\n2. STATUT BCG SELON h2 (détail)")
for val, lbl in [(1,'Vacciné BCG (h2=1 ou 2)'), (0,'Non vacciné (h2=0)')]:
    n = (df_analyse['bcg'] == val).sum()
    print(f"   {lbl:<30} : {n:,} ({n/N*100:.1f}%)")

# Délai jusqu'à la vaccination ou censure
print(f"\n3. DÉLAI JUSQU'AU BCG OU CENSURE (mois)")
print(f"   Médiane  : {df_analyse['duree'].median():.1f} mois")
print(f"   Moyenne  : {df_analyse['duree'].mean():.1f} mois")
print(f"   Min–Max  : {df_analyse['duree'].min():.1f} – {df_analyse['duree'].max():.1f} mois")
print(f"   ≤ 1 mois : {(df_analyse['duree'] <= 1).sum():,} ({(df_analyse['duree'] <= 1).mean()*100:.1f}%)")
print(f"   ≤ 3 mois : {(df_analyse['duree'] <= 3).sum():,} ({(df_analyse['duree'] <= 3).mean()*100:.1f}%)")

# Export
desc_data = {
    'Indicateur': ['N total', 'Vaccinés BCG — date connue (event=1)', 'Censurés (event=0)',
                   'BCG final (h2=1 ou 2)', 'Non vaccinés (h2=0)', 'Délai médian (mois)'],
    'Valeur': [N, n_vaccines, n_non_vaccines,
               (df_analyse['bcg'] == 1).sum(), (df_analyse['bcg'] == 0).sum(),
               df_analyse['duree'].median()],
}
pd.DataFrame(desc_data).to_csv("../outputs/tables/01_description_echantillon.csv", index=False)
print("\n✓ Sauvegardé → 01_description_echantillon.csv")

TABLEAU 1 — DESCRIPTION DE L'ÉCHANTILLON FINAL (N=4,765)

1. STATUT VACCINAL BCG (événement de survie)
   Vacciné BCG — date connue (event=1) : 3,251 (68.2%)
   Censuré — date inconnue   (event=0) : 1,514 (31.8%)

2. STATUT BCG SELON h2 (détail)
   Vacciné BCG (h2=1 ou 2)        : 4,026 (84.5%)
   Non vacciné (h2=0)             : 739 (15.5%)

3. DÉLAI JUSQU'AU BCG OU CENSURE (mois)
   Médiane  : 1.0 mois
   Moyenne  : 6.1 mois
   Min–Max  : 0.5 – 36.0 mois
   ≤ 1 mois : 2,919 (61.3%)
   ≤ 3 mois : 3,320 (69.7%)

✓ Sauvegardé → 01_description_echantillon.csv


In [14]:
# ── 4.2 Figure — Distribution de la durée de suivi ──────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Gauche : distribution du délai par statut d'événement
axes[0].hist(df_analyse[df_analyse['event']==0]['duree'], bins=30,
             color=BLUE, alpha=0.7, label='Censurés (date BCG inconnue)')
axes[0].hist(df_analyse[df_analyse['event']==1]['duree'], bins=30,
             color=GREEN, alpha=0.7, label='Vaccinés BCG (date connue)')
axes[0].set_xlabel("Délai depuis la naissance (mois)")
axes[0].set_ylabel("Nombre d'enfants")
axes[0].set_title("Distribution du délai jusqu'au BCG ou censure", fontweight='bold')
axes[0].legend()

# Droite : répartition vacciné / non-vacciné par statut d'événement
bcg_cross = pd.crosstab(df_analyse['bcg_label'], df_analyse['event'],
                         normalize='index') * 100
bcg_cross.columns = ['Censuré (%)', 'Vacciné — date connue (%)']
bcg_cross.plot(kind='bar', ax=axes[1], color=[BLUE, GREEN], edgecolor='white')
axes[1].set_title("Statut de l'événement selon BCG (h2)", fontweight='bold')
axes[1].set_xlabel("")
axes[1].set_ylabel("Pourcentage (%)")
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(loc='upper right')

for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%',
                     (p.get_x()+p.get_width()/2, p.get_height()+0.3),
                     ha='center', fontsize=9)

fig.suptitle("Description de l'échantillon — Premiers-nés EDSC-V 2018",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/fig1_description_echantillon.png", bbox_inches='tight')
plt.show()
print("✓ Figure 1 sauvegardée")

✓ Figure 1 sauvegardée


In [15]:
# ── 5.1 Kaplan-Meier global ──────────────────────────────
kmf_global = KaplanMeierFitter()
kmf_global.fit(
    durations  = df_analyse['duree'],
    event_observed = df_analyse['event'],
    label      = "Tous les premiers-nés"
)

print("Kaplan-Meier — Survie globale :")
print(f"  Médiane de survie : {kmf_global.median_survival_time_:.1f} mois")
print(f"  % non vacciné à 12 mois  : {kmf_global.predict(12):.4f} ({kmf_global.predict(12)*100:.2f}%)")
print(f"  % non vacciné à 24 mois  : {kmf_global.predict(24):.4f} ({kmf_global.predict(24)*100:.2f}%)")
print(f"  % non vacciné à 60 mois  : {kmf_global.predict(60):.4f} ({kmf_global.predict(60)*100:.2f}%)")

Kaplan-Meier — Survie globale :
  Médiane de survie : 1.0 mois
  % non vacciné à 12 mois  : 0.3062 (30.62%)
  % non vacciné à 24 mois  : 0.3038 (30.38%)
  % non vacciné à 60 mois  : 0.3031 (30.31%)


In [16]:
# ── 5.2 Kaplan-Meier par milieu de résidence ─────────────
# Question : les enfants urbains reçoivent-ils le BCG plus tôt que les ruraux ?

kmf_urbain = KaplanMeierFitter()
kmf_rural  = KaplanMeierFitter()

if 'residence' in df_analyse.columns and df_analyse['residence'].notna().sum() > 0:
    mask_urbain = df_analyse['residence'] == 'Urbain'
    mask_rural  = df_analyse['residence'] == 'Rural'
    n_urb = mask_urbain.sum()
    n_rur = mask_rural.sum()

    kmf_urbain.fit(
        durations      = df_analyse.loc[mask_urbain, 'duree'],
        event_observed = df_analyse.loc[mask_urbain, 'event'],
        label = f"Urbain (n={n_urb:,})"
    )
    kmf_rural.fit(
        durations      = df_analyse.loc[mask_rural, 'duree'],
        event_observed = df_analyse.loc[mask_rural, 'event'],
        label = f"Rural (n={n_rur:,})"
    )

    print("Kaplan-Meier par milieu de résidence :")
    print(f"\n  {'Indicateur':<30} {'Urbain':>12} {'Rural':>12}")
    print(f"  {'-'*56}")
    for t in [1, 3, 6, 12, 24]:
        s_urb = kmf_urbain.predict(t) * 100
        s_rur = kmf_rural.predict(t)  * 100
        diff  = s_urb - s_rur
        print(f"  S(t={t:>2} mois) non vacciné  {s_urb:>11.2f}% {s_rur:>11.2f}%  Δ={diff:+.2f}%")

    print(f"\n  Médiane délai — Urbain : {kmf_urbain.median_survival_time_:.1f} mois")
    print(f"  Médiane délai — Rural  : {kmf_rural.median_survival_time_:.1f} mois")
else:
    print("⚠ Variable résidence (v025) absente — exécuter d'abord la cellule 1.1b")
    # Groupes fictifs pour éviter les erreurs dans les cellules suivantes
    mask_urbain = pd.Series([False] * len(df_analyse))
    mask_rural  = pd.Series([False] * len(df_analyse))

Kaplan-Meier par milieu de résidence :

  Indicateur                           Urbain        Rural
  --------------------------------------------------------
  S(t= 1 mois) non vacciné        40.91%       40.72%  Δ=+0.20%
  S(t= 3 mois) non vacciné        34.57%       33.03%  Δ=+1.54%
  S(t= 6 mois) non vacciné        33.23%       30.63%  Δ=+2.60%
  S(t=12 mois) non vacciné        31.86%       29.22%  Δ=+2.64%
  S(t=24 mois) non vacciné        31.70%       28.94%  Δ=+2.75%

  Médiane délai — Urbain : 1.0 mois
  Médiane délai — Rural  : 1.0 mois


In [17]:
# ── 5.3 Test du Log-rank — Urbain vs Rural ───────────────
if mask_urbain.sum() > 0 and mask_rural.sum() > 0:
    results_logrank = logrank_test(
        durations_A      = df_analyse.loc[mask_urbain, 'duree'],
        durations_B      = df_analyse.loc[mask_rural,  'duree'],
        event_observed_A = df_analyse.loc[mask_urbain, 'event'],
        event_observed_B = df_analyse.loc[mask_rural,  'event'],
    )

    print("=" * 55)
    print("TEST DU LOG-RANK — Urbain vs Rural")
    print("=" * 55)
    print(f"  Statistique du test (χ²) : {results_logrank.test_statistic:.4f}")
    print(f"  p-value                  : {results_logrank.p_value:.6f}")
    print(f"  Degrés de liberté        : 1")
    print()
    if results_logrank.p_value < 0.001:
        print("  → *** Différence très hautement significative (p<0,001)")
        print("  → Le délai de vaccination BCG diffère entre urbain et rural")
    elif results_logrank.p_value < 0.05:
        print("  → *  Différence significative (p<0,05)")
    else:
        print("  → NS — Pas de différence significative entre les groupes")

    pd.DataFrame({
        'Test': ['Log-rank Urbain vs Rural'],
        'Statistique χ²': [round(results_logrank.test_statistic, 4)],
        'p-value': [round(results_logrank.p_value, 6)],
        'Significativité': ['***' if results_logrank.p_value < 0.001 else
                            ('*' if results_logrank.p_value < 0.05 else 'NS')]
    }).to_csv("../outputs/tables/02_logrank_test.csv", index=False)
    print("\n✓ Sauvegardé → 02_logrank_test.csv")
else:
    print("⚠ Test log-rank non disponible : variable résidence absente")

TEST DU LOG-RANK — Urbain vs Rural
  Statistique du test (χ²) : 2.7850
  p-value                  : 0.095153
  Degrés de liberté        : 1

  → NS — Pas de différence significative entre les groupes

✓ Sauvegardé → 02_logrank_test.csv


In [18]:
# ── 5.4 Figure — Courbes de Kaplan-Meier ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Graphique gauche : KM par résidence (si disponible)
ax1 = axes[0]
if mask_urbain.sum() > 0:
    kmf_urbain.plot_survival_function(ax=ax1, color=BLUE, linewidth=2.2, ci_show=True)
    kmf_rural.plot_survival_function(ax=ax1, color=AMBER, linewidth=2.2,
                                      linestyle='--', ci_show=True)
    ax1.set_title("Courbes de Kaplan-Meier\nDélai jusqu'au BCG par résidence",
                  fontweight='bold', fontsize=11)
    # Annotation p-value
    if 'results_logrank' in dir():
        pval_txt = ("Log-rank p < 0,001 ***" if results_logrank.p_value < 0.001
                    else f"Log-rank p={results_logrank.p_value:.4f}")
        ax1.text(0.98, 0.08, pval_txt, transform=ax1.transAxes,
                 ha='right', fontsize=10, color='black',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray'))
else:
    kmf_global.plot_survival_function(ax=ax1, color=NAVY, linewidth=2.5, ci_show=True)
    ax1.set_title("Courbe de Kaplan-Meier globale", fontweight='bold', fontsize=11)

ax1.axvline(x=12, color=GRAY, linestyle=':', alpha=0.6, label='12 mois')
ax1.set_xlabel("Temps depuis la naissance (mois)")
ax1.set_ylabel("Probabilité de non-vaccination S(t)")
ax1.set_ylim(0, 1.05)
ax1.legend(loc='upper right')

# Graphique droit : KM global avec repère médiane
ax2 = axes[1]
kmf_global.plot_survival_function(ax=ax2, color=NAVY, linewidth=2.5, ci_show=True,
                                   label=f"Tous premiers-nés (n={len(df_analyse):,})")
ax2.axhline(y=0.5, color=GRAY, linestyle=':', alpha=0.6, label='S(t) = 50%')
med = kmf_global.median_survival_time_
ax2.axvline(x=med, color=GREEN, linestyle=':', alpha=0.8,
            label=f"Médiane = {med:.0f} mois")
ax2.set_title("Courbe de Kaplan-Meier globale\nDélai médian jusqu'au BCG",
              fontweight='bold', fontsize=11)
ax2.set_xlabel("Temps depuis la naissance (mois)")
ax2.set_ylabel("Probabilité de non-vaccination S(t)")
ax2.set_ylim(0, 1.05)
ax2.legend(loc='upper right')

fig.suptitle("Analyse de Kaplan-Meier — Temps jusqu'au BCG\n"
             "Premiers-nés EDSC-V Cameroun 2018",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/fig2_kaplan_meier_bcg.png", bbox_inches='tight')
plt.show()
print("✓ Figure 2 (Kaplan-Meier) sauvegardée")

✓ Figure 2 (Kaplan-Meier) sauvegardée


In [19]:
# ── 5.5 KM par sous-groupes (résidence, sexe) ────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Gauche : par résidence
for grp, col in [('Urbain', BLUE), ('Rural', AMBER)]:
    if 'residence' in df_analyse.columns:
        mask_grp = df_analyse['residence'] == grp
        if mask_grp.sum() > 10:
            kmf_tmp = KaplanMeierFitter()
            kmf_tmp.fit(
                df_analyse.loc[mask_grp,'duree'],
                df_analyse.loc[mask_grp,'event'],
                label=grp
            )
            kmf_tmp.plot_survival_function(ax=axes[0], color=col, linewidth=2)
axes[0].set_title("KM par milieu de résidence", fontweight='bold')
axes[0].set_xlabel("Temps (mois)")
axes[0].set_ylabel("Probabilité de survie S(t)")
axes[0].legend()

# Droite : par sexe
for grp, col in [('Masculin', NAVY), ('Féminin', CORAL)]:
    if 'sexe' in df_analyse.columns:
        mask_grp = df_analyse['sexe'] == grp
        if mask_grp.sum() > 10:
            kmf_tmp = KaplanMeierFitter()
            kmf_tmp.fit(
                df_analyse.loc[mask_grp,'duree'],
                df_analyse.loc[mask_grp,'event'],
                label=grp
            )
            kmf_tmp.plot_survival_function(ax=axes[1], color=col, linewidth=2)
axes[1].set_title("KM par sexe de l'enfant", fontweight='bold')
axes[1].set_xlabel("Temps (mois)")
axes[1].set_ylabel("Probabilité de survie S(t)")
axes[1].legend()

fig.suptitle("Courbes de Kaplan-Meier par sous-groupes — Premiers-nés EDSC-V 2018",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("../outputs/figures/fig3_km_sousgroupes.png", bbox_inches='tight')
plt.show()
print("✓ Figure 3 (KM sous-groupes) sauvegardée")

✓ Figure 3 (KM sous-groupes) sauvegardée


In [20]:
# ── 6.1 Préparation des données pour Cox ─────────────────
# NB : bcg (h2) est EXCLU du modèle — il est quasi-identique à event (séparation parfaite)
# On utilise uniquement les facteurs socioéconomiques comme prédicteurs
vars_cox = ['duree', 'event']
vars_covariables = []

for var, label in [('v025', 'milieu_residence'), ('v106', 'instruction_mere'),
                   ('v190', 'quintile_richesse'), ('b4',  'sexe_enfant'),
                   ('v012', 'age_mere')]:
    if var in df_analyse.columns:
        df_analyse[label] = df_analyse[var]
        vars_covariables.append(label)

df_cox = df_analyse[vars_cox + vars_covariables].dropna().copy()

# Encodage : sexe_enfant 1=masculin → 0, 2=féminin → 1
if 'sexe_enfant' in df_cox.columns:
    df_cox['sexe_enfant'] = (df_cox['sexe_enfant'] == 2).astype(int)

print(f"Données pour le modèle de Cox :")
print(f"  N observations              : {len(df_cox):,}")
print(f"  Événements (BCG date connue): {df_cox['event'].sum():,}")
print(f"  Covariables socioéconomiques: {vars_covariables}")
print(f"  Taux d'événement            : {df_cox['event'].mean()*100:.2f}%")
if not vars_covariables:
    print("\n  ⚠ Aucune covariable disponible — exécuter d'abord la cellule 1.1b (extraction SAV)")

Données pour le modèle de Cox :
  N observations              : 4,645
  Événements (BCG date connue): 3,174
  Covariables socioéconomiques: ['milieu_residence', 'instruction_mere', 'quintile_richesse', 'sexe_enfant', 'age_mere']
  Taux d'événement            : 68.33%


In [21]:
# ── 6.3 Modèle de Cox multivarié (ajusté) ────────────────
if not vars_covariables:
    print("⚠ Covariables absentes — exécuter d'abord la cellule 1.1b (extraction SAV)")
    hr_table = pd.DataFrame(columns=['Variable','HR','IC inf.','IC sup.','p-value','Sig.'])
else:
    vars_modele  = ['duree', 'event'] + vars_covariables
    df_cox_multi = df_cox[vars_modele].dropna()

    cph_multi = CoxPHFitter()
    cph_multi.fit(df_cox_multi, duration_col='duree', event_col='event')

    print("=" * 60)
    print("MODÈLE DE COX MULTIVARIÉ — Facteurs socioéconomiques")
    print("Variable dépendante : délai jusqu'au BCG (en mois)")
    print("=" * 60)
    cph_multi.print_summary(decimals=4)

    hr_table = pd.DataFrame({
        'Variable': cph_multi.summary.index,
        'HR'      : np.exp(cph_multi.params_).values.round(4),
        'IC inf.' : np.exp(cph_multi.confidence_intervals_['95% lower-bound']).values.round(4),
        'IC sup.' : np.exp(cph_multi.confidence_intervals_['95% upper-bound']).values.round(4),
        'p-value' : cph_multi.summary['p'].values.round(4),
    })
    hr_table['Sig.'] = hr_table['p-value'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'NS'))
    )
    hr_table.to_csv("../outputs/tables/03_cox_hazard_ratios.csv", index=False)
    print("\n✓ Sauvegardé → 03_cox_hazard_ratios.csv")
    print("\nInterprétation des HR :")
    print("  HR > 1 → ce facteur ACCÉLÈRE la vaccination BCG (risque plus élevé d'être vacciné)")
    print("  HR < 1 → ce facteur RETARDE  la vaccination BCG")

MODÈLE DE COX MULTIVARIÉ — Facteurs socioéconomiques
Variable dépendante : délai jusqu'au BCG (en mois)


<lifelines.CoxPHFitter: fitted with 4645 total observations, 1471 right-censored observations>
             duration col = 'duree'
                event col = 'event'
      baseline estimation = breslow
   number of observations = 4645
number of events observed = 3174
   partial log-likelihood = -25235.0366
         time fit was run = 2026-06-19 23:24:10 UTC

---
                     coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                            
milieu_residence  -0.0148    0.9853    0.0473         -0.1076          0.0779              0.8980              1.0810
instruction_mere   0.0136    1.0137    0.0252         -0.0358          0.0631              0.9649              1.0651
quintile_richesse -0.0363    0.9643    0.0225         -0.0805          0.0078              0.9227              1.0078
sexe_enfant        0.0120    1.0121    0.0362         -0.0589          0.0828              0.9428              1.0864
age_mere          -0.0123    0.9878    0.0044         -0.0210         -0.0036              0.9792              0.9964

                   cmp to       z      p  -log2(p)
covariate                                         
milieu_residence   0.0000 -0.3137 0.7538    0.4078
instruction_mere   0.0000  0.5413 0.5883    0.7653
quintile_richesse  0.0000 -1.6129 0.1068    3.2276
sexe_enfant        0.0000  0.3313 0.7404    0.4337
age_mere           0.0000 -2.7770 0.0055    7.5099
---
Concordance = 0.5212
Partial AIC = 50480.0732
log-likelihood ratio test = 16.9364 on 5 df
-log2(p) of ll-ratio test = 7.7573


✓ Sauvegardé → 03_cox_hazard_ratios.csv

Interprétation des HR :
  HR > 1 → ce facteur ACCÉLÈRE la vaccination BCG (risque plus élevé d'être vacciné)
  HR < 1 → ce facteur RETARDE  la vaccination BCG


In [22]:
# ── 6.4 Vérification hypothèse risques proportionnels ─────
# Test de Schoenfeld : p > 0,05 → hypothèse vérifiée
print("=" * 55)
print("TEST DES RÉSIDUS DE SCHOENFELD")
print("(Vérification risques proportionnels)")
print("=" * 55)
print()

if 'cph_multi' not in dir() or 'df_cox_multi' not in dir():
    print("⚠ Modèle Cox non disponible — exécuter d'abord les cellules 1.1b et 6.3")
else:
    cph_multi.check_assumptions(df_cox_multi, p_value_threshold=0.05, show_plots=False)

TEST DES RÉSIDUS DE SCHOENFELD
(Vérification risques proportionnels)

Proportional hazard assumption looks okay.


In [23]:
# ── 6.5 Figure — Forest Plot des Hazard Ratios ───────────
if hr_table.empty:
    print("⚠ Forest plot non disponible : covariables absentes")
else:
    fig, ax = plt.subplots(figsize=(10, max(4, len(hr_table) * 0.8 + 1.5)))

    y_pos = np.arange(len(hr_table))
    colors_hr = [GREEN if (r['HR'] > 1 and r['Sig.'] != 'NS') else
                 (CORAL if (r['HR'] < 1 and r['Sig.'] != 'NS') else GRAY)
                 for _, r in hr_table.iterrows()]

    for i, (_, row) in enumerate(hr_table.iterrows()):
        ax.errorbar(
            row['HR'], i,
            xerr=[[row['HR'] - row['IC inf.']], [row['IC sup.'] - row['HR']]],
            fmt='o', color=colors_hr[i], ecolor=colors_hr[i],
            capsize=5, markersize=9, linewidth=2
        )
        ax.text(
            hr_table['IC sup.'].max() * 1.05, i,
            f"HR={row['HR']:.3f}  {row['Sig.']}",
            va='center', fontsize=9
        )

    ax.axvline(x=1, color='black', linestyle='--', linewidth=1.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(hr_table['Variable'], fontsize=10)
    ax.set_xlabel("Hazard Ratio (IC 95%)", fontsize=11)
    ax.set_title("Forest Plot — Modèle de Cox ajusté\n"
                 "Facteurs socioéconomiques du délai de vaccination BCG\n"
                 "Premiers-nés EDSC-V Cameroun 2018",
                 fontsize=12, fontweight='bold')

    import matplotlib.patches as mpatches
    p1 = mpatches.Patch(color=GREEN, label='Accélère la vaccination (HR>1, p<0,05)')
    p2 = mpatches.Patch(color=CORAL, label='Retarde la vaccination (HR<1, p<0,05)')
    p3 = mpatches.Patch(color=GRAY,  label='Non significatif')
    ref = plt.Line2D([0],[0], color='black', linestyle='--', label='Référence HR=1')
    ax.legend(handles=[p1, p2, p3, ref], loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.savefig("../outputs/figures/fig4_cox_forest_plot.png", bbox_inches='tight')
    plt.show()
    print("✓ Figure 4 (Forest Plot Cox) sauvegardée")

✓ Figure 4 (Forest Plot Cox) sauvegardée


In [24]:
# ── Résumé final ─────────────────────────────────────────
print("=" * 60)
print("ANALYSE DE SURVIE — RÉSUMÉ FINAL")
print("=" * 60)
print(f"\nÉchantillon   : {len(df_analyse):,} premiers-nés (EDSC-V 2018)")
print(f"Événements    : {df_analyse['event'].sum():,} vaccinations BCG (date connue)")
print(f"Censurés      : {(df_analyse['event']==0).sum():,} (BCG non daté ou non reçu)")

print(f"\nKaplan-Meier global :")
print(f"  Médiane (non vacciné)    : {kmf_global.median_survival_time_:.1f} mois")
print(f"  S(12 mois)               : {kmf_global.predict(12)*100:.2f}% non vaccinés à 12 mois")
print(f"  S(60 mois)               : {kmf_global.predict(60)*100:.2f}% non vaccinés à 60 mois")

# KM résidence — disponible seulement si kmf_urbain a été ajusté
if 'kmf_urbain' in dir() and hasattr(kmf_urbain, 'median_survival_time_'):
    print(f"\nKaplan-Meier par résidence :")
    print(f"  Médiane — Urbain : {kmf_urbain.median_survival_time_:.1f} mois")
    print(f"  Médiane — Rural  : {kmf_rural.median_survival_time_:.1f} mois")
    if 'results_logrank' in dir():
        sig = ('***' if results_logrank.p_value < 0.001 else
               ('*' if results_logrank.p_value < 0.05 else 'NS'))
        print(f"  Log-rank p-value : {results_logrank.p_value:.6f} {sig}")
else:
    print(f"\n⚠ KM résidence : variable v025 non chargée (exécuter cellule 1.1b)")

# Modèle Cox
if 'hr_table' in dir() and not hr_table.empty:
    print(f"\nModèle de Cox ajusté — Facteurs socioéconomiques :")
    for _, row in hr_table.iterrows():
        direction = "→ accélère BCG" if row['HR'] > 1 else "→ retarde BCG"
        print(f"  {row['Variable']:<25} HR={row['HR']:.3f}  "
              f"IC95%[{row['IC inf.']:.3f}–{row['IC sup.']:.3f}]  {row['Sig.']:>3}  {direction}")
else:
    print(f"\n⚠ Modèle de Cox : exécuter la cellule 1.1b pour charger les covariables")

print(f"\nFichiers générés dans outputs/tables/ et outputs/figures/")

ANALYSE DE SURVIE — RÉSUMÉ FINAL

Échantillon   : 4,765 premiers-nés (EDSC-V 2018)
Événements    : 3,251 vaccinations BCG (date connue)
Censurés      : 1,514 (BCG non daté ou non reçu)

Kaplan-Meier global :
  Médiane (non vacciné)    : 1.0 mois
  S(12 mois)               : 30.62% non vaccinés à 12 mois
  S(60 mois)               : 30.31% non vaccinés à 60 mois

Kaplan-Meier par résidence :
  Médiane — Urbain : 1.0 mois
  Médiane — Rural  : 1.0 mois
  Log-rank p-value : 0.095153 NS

Modèle de Cox ajusté — Facteurs socioéconomiques :
  milieu_residence          HR=0.985  IC95%[0.898–1.081]   NS  → retarde BCG
  instruction_mere          HR=1.014  IC95%[0.965–1.065]   NS  → accélère BCG
  quintile_richesse         HR=0.964  IC95%[0.923–1.008]   NS  → retarde BCG
  sexe_enfant               HR=1.012  IC95%[0.943–1.086]   NS  → accélère BCG
  age_mere                  HR=0.988  IC95%[0.979–0.996]   **  → retarde BCG

Fichiers générés dans outputs/tables/ et outputs/figures/
